# 7 Qwen3-8B 图模式：GE Graph 与 NPU Graph EX

本章参考第 4 章的 A/B 验证方式，在相同模型、输入、batch、输出长度和算子开关下，对比 `eager`、`ge_graph` 与 `npugraph_ex` 三种执行模式，重点观察 GE Graph 和 NPU Graph EX 的性能差异。当前 recipes 实现中，**两种图模式都只编译 decode，prefill 仍使用 Eager**；首次 warm-up 会触发编译，因此要同时观察正式推理吞吐、decode 耗时与首轮端到端耗时。

## 1 环境与工作区准备

本章不安装依赖，直接复用第 2 章检查完成的 CANN、PyTorch、torch_npu 与 TorchAir 环境。模型权重继续使用默认缓存；如已准备本地权重，可在启动 Notebook 前设置 `QWEN3_8B_MODEL_PATH`。


In [ ]:
# 标准库：处理 JSON、环境变量、子进程、模块路径、计时和文件路径。
import json
import os
import subprocess
import sys
import time
from pathlib import Path

print('[环境准备] 开始', flush=True)
# 默认从 ModelScope 获取模型；如果外部已设置这些变量，则保留外部配置。
os.environ.setdefault('QWEN3_8B_DOWNLOAD_BACKEND', 'modelscope')
# 关闭 tokenizer 并行提示，避免多进程推理日志中出现无关警告。
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# 默认只使用 0 号 NPU，也可以在启动 Notebook 前覆盖。
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')

def locate_repo_root():
    """从常用环境变量和当前目录的父目录中定位仓库根目录。"""
    candidates = []
    # CANNLab、GitCode 和本地环境可能使用不同的仓库根目录变量。
    for env_name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(env_name):
            candidates.append(Path(os.environ[env_name]))
    try:
        # Notebook 可能从仓库内任意子目录启动，因此逐级向上查找。
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        candidates.extend(lineage)
        candidates.extend(base / 'cann-learning-hub' for base in lineage)
    except FileNotFoundError:
        pass
    # 在线环境的常见兜底路径。
    candidates.append(Path('/opt/atomgit/cann-learning-hub'))
    # dict.fromkeys 在去重的同时保留原有候选顺序。
    for candidate in dict.fromkeys(map(str, candidates)):
        root = Path(candidate)
        if (root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return root
    raise FileNotFoundError('无法定位 cann-learning-hub，请在仓库中打开 Notebook，或设置 CANN_LEARNING_HUB_ROOT。')

# 将工作目录统一到仓库根目录，避免后续相对路径随 Notebook 启动位置变化。
REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
SRC_DIR = TUTORIAL_DIR / 'src'
SOURCE_DIR = REPO_ROOT / 'Sources' / 'model_inference_optimization' / 'qwen3_8b'
# Sources 目录只存运行时 YAML、日志和指标，不修改课程模板。
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
# 把教程 src 加入模块搜索路径，以便导入本课程的辅助函数。
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 复用前面章节准备的统一 Python 依赖；这里不主动安装新包。
from inference_scripts.environment_check import require_unified_environment
require_unified_environment(TUTORIAL_DIR / 'requirements.txt')
if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME 未设置，请先激活 CANN 环境。')
# Notebook 内核不会自动继承终端 source 的全部变量，因此读取 set_env.sh 的结果并回填。
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env = subprocess.check_output(f"bash -lc 'source {cann_script} && env'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

# 三份 YAML 只允许 exe_mode 不同，用于公平比较执行后端。
RECIPE_ROOT = SRC_DIR / 'inference_scripts' / 'recipe_qwen3_8b'
RECIPE_CONFIG_DIR = RECIPE_ROOT / 'models' / 'qwen' / 'config'
EAGER_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_8b_1tp.yaml'
GE_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_8b_1tp_ge_graph.yaml'
NPUGRAPH_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_8b_1tp_npugraph_ex.yaml'
# 优先使用用户指定的本地权重，否则使用模型仓标识并复用下载缓存。
MODEL_ID = os.environ.get('QWEN3_8B_MODEL_PATH') or 'Qwen/Qwen3-8B'

print('仓库目录 =', REPO_ROOT)
print('运行目录 =', SOURCE_DIR)
print('recipes 目录 =', RECIPE_ROOT)
print('Eager YAML =', EAGER_YAML_TEMPLATE)
print('GE Graph YAML =', GE_YAML_TEMPLATE)
print('NPU Graph EX YAML =', NPUGRAPH_YAML_TEMPLATE)
print('可见 NPU =', os.environ.get('ASCEND_RT_VISIBLE_DEVICES'))
print('[环境准备] 完成', flush=True)


## 2 理解开关与 A/B 口径

核心开关位于 YAML 的 `model_config`：

```yaml
exe_mode: "ge_graph"       # TorchAir GE Graph
# 或 exe_mode: "npugraph_ex" # NPU Graph EX
enable_cache_compile: False
```

`ge_graph` 通过 TorchAir GE 后端以静态 shape 编译 decode；`npugraph_ex` 使用 NPU Graph EX 后端的动态图编译路径，recipes 会为它自动设置 `TASK_QUEUE_ENABLE=1`。本章令 `enable_cache_compile=False`、`enable_static_kernel=False`，只比较后端本身；Dense RMSNorm 融合、Profiler、数据集、输入截断长度、batch、输出长度和 thinking 开关均保持一致。


### GE Graph 与 NPU Graph EX：关系与差异

两种图模式解决的是同一个问题：Eager 模式下每个算子都由 Host 侧逐个下发，而 Decode 阶段每步只处理 1 个 token、单算子计算量小，Host 下发与调度开销占比很高。因此两种模式在 recipes 中的接入方式完全对称：都通过 `torch.compile(..., fullgraph=True)` 走 TorchAir 集成（`executor/utils/graph_utils.py`），都只编译 Decode、Prefill 保持 Eager（`executor/core/model_worker/model_worker.py` 中的 `if self.exe_mode in ["ge_graph", "npugraph_ex"] and not is_prefill` 分支），也都支持 `enable_cache_compile` 编译缓存。差异集中在“图怎么建、怎么执行”：

| 维度 | GE Graph（`ge_graph`） | NPU Graph EX（`npugraph_ex`） |
|------|------------------------|-------------------------------|
| 技术路线 | TorchAir 把 Decode 子图翻译为 GE（Graph Engine）计算图并提前整图编译：算子融合、tiling 调度、拓扑排序等优化在编译期完成，运行时整图一次性下发 | 基于 NPU Graph 的增强后端（“EX”），思路与 CUDA Graph 类似：把算子下发序列固化为可整图重放的执行单元，消除逐算子的 Host 下发开销 |
| Shape 处理 | `dynamic=False`，静态 shape 编译，shape 变化需重新编译 | `dynamic=True`，动态图路径，对 shape 变化更友好 |
| 编译配置 | `CompilerConfig` 携带 `tiling_schedule_optimize`、`topology_sorting_strategy` 等 GE 级优化选项 | `options` 携带 `static_kernel_compile` 等，`enable_static_kernel` 仅此模式支持 |
| 任务队列 | 沿用默认 `TASK_QUEUE_ENABLE=2`（Host 性能优先） | recipes 自动改设 `TASK_QUEUE_ENABLE=1`（该后端仅支持 0 或 1） |
| 配套功能 | `enable_prefetch` 权重预取当前仅支持 `ge_graph` | — |

对性能的典型影响（本章 A/B 会用同一工作负载量化验证）：

- **GE Graph**：编译做得深，首次 warm-up 编译耗时长、冷启动 wall time 明显更长，换来稳态 Decode 吞吐通常最高。
- **NPU Graph EX**：编译路径更轻，冷启动明显更快，稳态吞吐通常介于 Eager 与 GE Graph 之间。
- 两者相对 Eager 都应观察到吞吐提升与 Decode 耗时下降；若生成文本相似度异常，应优先排查图模式引入的数值差异。

选型直觉：追求稳态吞吐且能接受较长的首次编译，选 GE Graph；对冷启动（首响时间）敏感或输入 shape 多变，选 NPU Graph EX。另外，两种模式 warm-up 时都会输出 `Try to compile model` / `triggering graph compilation` 编译日志，因此本章后续对两种模式共用同一套 `verify_graph_run` 证据检查，确认图编译确实发生。

In [ ]:
import yaml
from difflib import unified_diff

def ab_focus(path):
    """读取会影响三组实验公平性的 YAML 字段。"""
    config = yaml.safe_load(path.read_text(encoding='utf-8'))
    model = config['model_config']
    return {
        'exe_mode': model['exe_mode'],
        'enable_cache_compile': model['enable_cache_compile'],
        'enable_profiler': model['enable_profiler'],
        'custom_params': model['custom_params'],
        'data_config': config['data_config'],
        'scheduler_config': config['scheduler_config'],
        'parallel_config': config['parallel_config'],
    }

# 分别提取 Eager、GE Graph 和 NPU Graph EX 的关键配置。
eager_focus = ab_focus(EAGER_YAML_TEMPLATE)
ge_focus = ab_focus(GE_YAML_TEMPLATE)
npugraph_focus = ab_focus(NPUGRAPH_YAML_TEMPLATE)
# 先确认每份模板指向预期后端，避免误用 YAML 导致结论失真。
assert eager_focus['exe_mode'] == 'eager'
assert ge_focus['exe_mode'] == 'ge_graph'
assert npugraph_focus['exe_mode'] == 'npugraph_ex'
# 去掉唯一允许变化的 exe_mode，再断言其余参数完全相同。
without_mode = lambda config: {key: value for key, value in config.items() if key != 'exe_mode'}
assert without_mode(eager_focus) == without_mode(ge_focus) == without_mode(npugraph_focus)
print('Eager 配置 =', json.dumps(eager_focus, ensure_ascii=False, indent=2))
print('GE Graph 配置 =', json.dumps(ge_focus, ensure_ascii=False, indent=2))
print('NPU Graph EX 配置 =', json.dumps(npugraph_focus, ensure_ascii=False, indent=2))
print('三组配置检查通过：除 exe_mode 外，其余关键字段一致。')

def show_diff(name_a, config_a, name_b, config_b):
    left = json.dumps(
        without_mode(config_a), ensure_ascii=False, indent=2, sort_keys=True
    ).splitlines()
    right = json.dumps(
        without_mode(config_b), ensure_ascii=False, indent=2, sort_keys=True
    ).splitlines()

    print("\n".join(unified_diff(
        left,
        right,
        fromfile=name_a,
        tofile=name_b,
        lineterm="",
    )))

print("Eager YAML =", EAGER_YAML_TEMPLATE.resolve())
print("GE YAML =", GE_YAML_TEMPLATE.resolve())
print("NPU Graph EX YAML =", NPUGRAPH_YAML_TEMPLATE.resolve())

show_diff("eager", eager_focus, "ge_graph", ge_focus)
show_diff("eager", eager_focus, "npugraph_ex", npugraph_focus)

## 3 准备公共运行函数

下面沿用第 4 章的 recipes 工作流：复制模板 YAML、填入实际模型路径、执行 `infer.sh`，再从 recipes 日志提取吞吐、时延与生成文本。`total_time_s` 来自正式推理的 prefill/decode 步骤日志，不含 warm-up；`wall_time_s` 则包含模型加载、warm-up 和首次编译等端到端开销。


In [ ]:
from inference_scripts.inference_qwen3_8b import ensure_recipe_log_ok
from inference_scripts.recipe_workflow import (
    collect_recipe_metrics,
    find_recipe_case_dir,
    prepare_runtime_yaml,
    print_yaml_focus,
    resolve_model_path,
)

def prepare_recipe_run(work_dir, yaml_path, template_path, log_name):
    """生成本轮运行 YAML，并把 Bash 单元需要的路径写入环境变量。"""
    # 每种模式使用独立目录，避免日志和指标互相覆盖。
    work_dir.mkdir(parents=True, exist_ok=True)
    # 将模型仓 ID 解析为可加载路径；已有本地权重或缓存时会直接复用。
    resolved_path, source = resolve_model_path(str(MODEL_ID), quiet_model_io=False)
    # 复制课程模板，仅把 model_path 替换为当前环境中的实际权重路径。
    prepare_runtime_yaml(template_path, yaml_path, model_path=resolved_path)
    print_yaml_focus(yaml_path)
    log_dir = work_dir / 'recipe_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    # %%bash 单元与 Python 内核共享环境变量，用它们传递运行目录、YAML 和日志路径。
    os.environ['QWEN3_RECIPE_ROOT'] = str(RECIPE_ROOT)
    os.environ['QWEN3_RUN_YAML'] = str(yaml_path)
    os.environ['QWEN3_RUN_LOG'] = str(log_dir / log_name)
    # 从启动命令前开始计时，用于计算包含加载、warm-up 和编译的 wall time。
    os.environ['QWEN3_RUN_START_TIME'] = str(time.perf_counter())
    return resolved_path, source

def collect_recipe_run(work_dir, tag, yaml_path, resolved_path, source, optimization_mode):
    """校验 recipes 日志，并把原始日志整理为统一指标 JSON。"""
    # infer.sh 按日期和 YAML 文件名创建结果目录，这里定位刚完成的 case。
    case_dir = find_recipe_case_dir(RECIPE_ROOT, yaml_path)
    if case_dir is None:
        raise RuntimeError(f'未找到 recipes 结果目录：{yaml_path.stem}')
    recipe_log_path = case_dir / 'log_0.log'
    # 先检查日志中是否存在报错或缺失结果，失败时停止后续性能比较。
    ensure_recipe_log_ok(recipe_log_path)
    # run_info 同时保存 recipes 内部日志和 Notebook 捕获的完整 stdout。
    run_info = {
        'stdout_path': os.environ['QWEN3_RUN_LOG'],
        'stderr_path': os.environ['QWEN3_RUN_LOG'],
        'wall_time_s': time.perf_counter() - float(os.environ['QWEN3_RUN_START_TIME']),
        'recipe_case_dir': str(case_dir),
        'recipe_log_path': str(recipe_log_path),
    }
    # 三组实验固定输出 64 token、输入上限 256，并关闭 thinking。
    return collect_recipe_metrics(
        RECIPE_ROOT, work_dir, tag, yaml_path, run_info,
        model_id=str(MODEL_ID),
        resolved_model_path=resolved_path,
        model_source=source,
        optimization_mode=optimization_mode,
        max_new_tokens=64,
        max_input_tokens=256,
        enable_thinking=False,
    )

def show_metrics(metrics):
    """打印最重要的性能与正确性指标，便于快速检查单次运行。"""
    print('执行模式 =', metrics.get('recipe_exe_mode'))
    print('生成 token 数 =', metrics.get('generated_tokens'))
    print('正式推理计时(s, prefill+decode) =', metrics.get('total_time_s'))
    print('其中 decode 耗时(s) =', metrics.get('decode_time_s'))
    # wall time 包含模型加载、warm-up 和首次图编译，不等同于稳态推理耗时。
    print('端到端 wall time(s) =', metrics.get('wall_time_s'))
    print('吞吐(tokens/s) =', metrics.get('tokens_per_second'))
    print('平均 token 时延(s) =', metrics.get('mean_per_token_latency_s'))
    print('吐字检查 =', (metrics.get('generation_text_sanity') or {}).get('status'))
    print('指标文件目录 =', Path(metrics['outputs_path']).parent.parent / 'metrics')


## 4 运行 Eager Baseline

先准备 `exe_mode=eager` 的运行 YAML。该结果作为图模式的对照组。


In [ ]:
# Eager 是未启用图编译的对照组。
EAGER_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'benchmark_graph_mode_eager'
# 运行 YAML 写到 Sources，课程目录中的模板保持不变。
EAGER_YAML = SOURCE_DIR / 'recipe_yaml' / 'graph_mode_eager.yaml'
eager_model_path, eager_model_source = prepare_recipe_run(
    EAGER_WORK_DIR, EAGER_YAML, EAGER_YAML_TEMPLATE, 'graph_mode_eager_stdout.log'
)


In [ ]:
%%bash
# -e：命令失败立即退出；-u：未定义变量报错；pipefail：管道中任一命令失败即失败。
set -euo pipefail
# 先打印实际命令，方便复现和排查传入了哪一份 YAML。
echo "cd ${QWEN3_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model qwen --mode offline --yaml ${QWEN3_RUN_YAML}"
# infer.sh 要从 recipes 根目录启动；tee 同时在界面显示并保存完整日志。
cd "${QWEN3_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model qwen --mode offline --yaml "${QWEN3_RUN_YAML}" 2>&1 | tee "${QWEN3_RUN_LOG}"


In [ ]:
# 从 Eager 日志提取性能和生成文本，供后面的三组对比使用。
eager_metrics = collect_recipe_run(
    EAGER_WORK_DIR, 'graph_mode_eager', EAGER_YAML,
    eager_model_path, eager_model_source, 'eager',
)
show_metrics(eager_metrics)


## 5 开启并运行 GE 图模式

图模式使用完全相同的启动命令，仅切换到 `exe_mode=ge_graph` 的 YAML。首次 decode warm-up 会进行图编译，等待时间通常明显长于 eager；这是预期行为，不要在编译期间中断进程。


In [ ]:
# GE Graph 组仅把 exe_mode 改为 ge_graph，其余参数与 Eager 一致。
GE_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'benchmark_graph_mode_ge'
GE_YAML = SOURCE_DIR / 'recipe_yaml' / 'graph_mode_ge.yaml'
ge_model_path, ge_model_source = prepare_recipe_run(
    GE_WORK_DIR, GE_YAML, GE_YAML_TEMPLATE, 'graph_mode_ge_stdout.log'
)


In [ ]:
%%bash
# 使用严格模式，确保编译或推理失败时不会继续生成无效指标。
set -euo pipefail
# 环境变量由上一个 Python 单元设置，此处只负责启动 recipes。
echo "cd ${QWEN3_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model qwen --mode offline --yaml ${QWEN3_RUN_YAML}"
# tee 保存的 stdout 还会用于确认 warm-up 是否真的触发图编译。
cd "${QWEN3_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model qwen --mode offline --yaml "${QWEN3_RUN_YAML}" 2>&1 | tee "${QWEN3_RUN_LOG}"


In [ ]:
# 解析 GE Graph 正式推理指标。
ge_metrics = collect_recipe_run(
    GE_WORK_DIR, 'graph_mode_ge', GE_YAML,
    ge_model_path, ge_model_source, 'ge_graph',
)
show_metrics(ge_metrics)

def verify_graph_run(metrics, expected_mode, work_dir):
    """同时检查 YAML 记录和编译日志，防止把 Eager 误认为图模式。"""
    # recipe_log 是框架日志，stdout_log 是 Notebook 通过 tee 保存的完整控制台输出。
    recipe_log = Path(metrics['recipe_log_path']).read_text(encoding='utf-8', errors='replace')
    stdout_log = Path(metrics['recipe_stdout_path']).read_text(encoding='utf-8', errors='replace')
    combined_log = recipe_log + '\n' + stdout_log
    # 不同日志配置可能只保留下面两个编译标记之一，因此使用“或”关系。
    evidence = {
        'expected_mode': expected_mode,
        'recorded_mode': metrics.get('recipe_exe_mode'),
        'yaml_mode_matches': metrics.get('recipe_exe_mode') == expected_mode,
        'warmup_trigger_seen': 'triggering graph compilation' in combined_log,
        'compile_request_seen': 'Try to compile model' in combined_log,
    }
    evidence['graph_mode_verified'] = (
        evidence['yaml_mode_matches']
        and (evidence['warmup_trigger_seen'] or evidence['compile_request_seen'])
    )
    # 将证据单独落盘，便于在性能结果之外审计后端是否真正启用。
    evidence_path = work_dir / 'metrics' / f'{expected_mode}_evidence.json'
    evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{expected_mode} 证据 =', json.dumps(evidence, ensure_ascii=False, indent=2))
    print('证据文件 =', evidence_path)
    return evidence

# 图编译证据是后续推荐后端的前置条件。
ge_evidence = verify_graph_run(ge_metrics, 'ge_graph', GE_WORK_DIR)
if not ge_evidence['graph_mode_verified']:
    raise RuntimeError('未在 YAML 与日志中同时确认 GE 图编译，请检查上方日志。')


## 6 开启并运行 NPU Graph EX

第三组切换到 `exe_mode=npugraph_ex`。该后端使用 `torch.compile(..., dynamic=True, fullgraph=True, backend="npugraph_ex")` 编译 decode，并由 recipes 将 `TASK_QUEUE_ENABLE` 调整为 1。本轮保持 `enable_static_kernel=False`，避免把 static kernel 的额外收益混入后端对比。


In [ ]:
# NPU Graph EX 组仍只改变 exe_mode，static kernel 和编译缓存均保持关闭。
NPUGRAPH_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'benchmark_graph_mode_npugraph_ex'
NPUGRAPH_YAML = SOURCE_DIR / 'recipe_yaml' / 'graph_mode_npugraph_ex.yaml'
npugraph_model_path, npugraph_model_source = prepare_recipe_run(
    NPUGRAPH_WORK_DIR, NPUGRAPH_YAML, NPUGRAPH_YAML_TEMPLATE, 'graph_mode_npugraph_ex_stdout.log'
)


In [ ]:
%%bash
# 严格模式可避免 npugraph_ex 编译失败后继续执行结果采集。
set -euo pipefail
# 打印实际命令，以确认本轮传入的是 NPU Graph EX YAML。
echo "cd ${QWEN3_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model qwen --mode offline --yaml ${QWEN3_RUN_YAML}"
# 推理输出既显示在 Notebook 中，也写入独立日志文件。
cd "${QWEN3_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model qwen --mode offline --yaml "${QWEN3_RUN_YAML}" 2>&1 | tee "${QWEN3_RUN_LOG}"


In [ ]:
# 采集 NPU Graph EX 指标，并复用同一套编译证据检查。
npugraph_metrics = collect_recipe_run(
    NPUGRAPH_WORK_DIR, 'graph_mode_npugraph_ex', NPUGRAPH_YAML,
    npugraph_model_path, npugraph_model_source, 'npugraph_ex',
)
show_metrics(npugraph_metrics)
npugraph_evidence = verify_graph_run(npugraph_metrics, 'npugraph_ex', NPUGRAPH_WORK_DIR)
if not npugraph_evidence['graph_mode_verified']:
    raise RuntimeError('未在 YAML 与日志中同时确认 NPU Graph EX 编译，请检查上方日志。')


## 7 对比三种模式并选择图后端

先用 Eager 检查两种图模式的基本收益，再直接计算 NPU Graph EX 相对 GE Graph 的吞吐变化、decode 耗时变化和冷启动变化。图模式可能带来轻微的 BF16 数值差异，因此还要检查三组吐字健康，以及 GE 与 NPU Graph EX 的文本相似度。建议在正式结论前重复运行多轮，并使用中位数减少抖动。


In [ ]:
# SequenceMatcher 用于检查不同后端生成文本是否保持足够一致。
from difflib import SequenceMatcher

def read_texts(metrics):
    """从 JSONL 生成结果中读取每条 prompt 对应的文本。"""
    rows = [json.loads(line) for line in Path(metrics['outputs_path']).read_text(encoding='utf-8').splitlines() if line.strip()]
    return [row.get('generated_text', '') for row in rows]

def similarity_ratio(left_texts, right_texts):
    """返回所有对应文本中的最低相似度，采用保守口径。"""
    if len(left_texts) != len(right_texts) or not left_texts:
        return 0.0
    return min(SequenceMatcher(None, left, right).ratio() for left, right in zip(left_texts, right_texts))

def gain_pct(new_value, old_value):
    """计算新方案相对旧方案的提升率；正数表示数值变大。"""
    return (new_value / old_value - 1.0) * 100 if old_value else None

def reduction_pct(new_value, old_value):
    """计算耗时/时延下降率；正数表示新方案更快。"""
    return (1.0 - new_value / old_value) * 100 if old_value and new_value is not None else None

def text_is_healthy(metrics):
    """确认生成文本未出现空输出、异常重复等健康度问题。"""
    return (metrics.get('generation_text_sanity') or {}).get('status') == 'ok'

# 正确性检查：既比较两种图后端，也分别与 Eager 对照。
eager_texts = read_texts(eager_metrics)
ge_texts = read_texts(ge_metrics)
npugraph_texts = read_texts(npugraph_metrics)
eager_ge_similarity = similarity_ratio(eager_texts, ge_texts)
eager_npugraph_similarity = similarity_ratio(eager_texts, npugraph_texts)
ge_npugraph_similarity = similarity_ratio(ge_texts, npugraph_texts)

# 性能检查：吞吐越大越好，平均时延和 decode 耗时越小越好。
eager_tps = eager_metrics['tokens_per_second']
ge_tps = ge_metrics['tokens_per_second']
npugraph_tps = npugraph_metrics['tokens_per_second']
eager_latency = eager_metrics['mean_per_token_latency_s']
ge_latency = ge_metrics['mean_per_token_latency_s']
npugraph_latency = npugraph_metrics['mean_per_token_latency_s']
eager_decode_time = eager_metrics.get('decode_time_s')
ge_decode_time = ge_metrics.get('decode_time_s')
npugraph_decode_time = npugraph_metrics.get('decode_time_s')
# 0.95 与第 4 章保持一致，允许 BF16 路径造成轻微文本差异。
all_text_healthy = all(text_is_healthy(metrics) for metrics in (eager_metrics, ge_metrics, npugraph_metrics))
graph_outputs_consistent = ge_npugraph_similarity >= 0.95
all_outputs_consistent = min(eager_ge_similarity, eager_npugraph_similarity, ge_npugraph_similarity) >= 0.95
# 只有后端确实编译成功且输出正确时，性能数字才参与推荐。
both_graph_modes_valid = (
    ge_evidence['graph_mode_verified']
    and npugraph_evidence['graph_mode_verified']
    and all_text_healthy
    and all_outputs_consistent
)
# 先选吞吐更高的图后端，再判断它是否真的优于 Eager。
if both_graph_modes_valid:
    preferred_graph_backend = 'npugraph_ex' if npugraph_tps > ge_tps else 'ge_graph'
    best_graph_tps = max(ge_tps, npugraph_tps)
    deployment_recommendation = (
        f'adopt_{preferred_graph_backend}' if best_graph_tps > eager_tps else 'keep_eager'
    )
else:
    preferred_graph_backend = 'not_determined'
    deployment_recommendation = 'investigate_before_adoption'

# 结构化保存原始指标、相对收益、文本验证和最终建议。
comparison = {
    'modes': {
        'eager': {
            'tokens_per_second': eager_tps,
            'mean_per_token_latency_s': eager_latency,
            'decode_time_s': eager_decode_time,
            'wall_time_s': eager_metrics.get('wall_time_s'),
        },
        'ge_graph': {
            'verified': ge_evidence['graph_mode_verified'],
            'tokens_per_second': ge_tps,
            'mean_per_token_latency_s': ge_latency,
            'decode_time_s': ge_decode_time,
            'wall_time_s': ge_metrics.get('wall_time_s'),
        },
        'npugraph_ex': {
            'verified': npugraph_evidence['graph_mode_verified'],
            'tokens_per_second': npugraph_tps,
            'mean_per_token_latency_s': npugraph_latency,
            'decode_time_s': npugraph_decode_time,
            'wall_time_s': npugraph_metrics.get('wall_time_s'),
        },
    },
    'relative_to_eager': {
        'ge_throughput_gain_pct': gain_pct(ge_tps, eager_tps),
        'ge_decode_time_reduction_pct': reduction_pct(ge_decode_time, eager_decode_time),
        'npugraph_ex_throughput_gain_pct': gain_pct(npugraph_tps, eager_tps),
        'npugraph_ex_decode_time_reduction_pct': reduction_pct(npugraph_decode_time, eager_decode_time),
    },
    # 这是本章重点：直接以 GE Graph 为基准衡量 NPU Graph EX。
    'npugraph_ex_relative_to_ge': {
        'throughput_gain_pct': gain_pct(npugraph_tps, ge_tps),
        'latency_reduction_pct': reduction_pct(npugraph_latency, ge_latency),
        'decode_time_reduction_pct': reduction_pct(npugraph_decode_time, ge_decode_time),
        'cold_start_wall_time_reduction_pct': reduction_pct(
            npugraph_metrics.get('wall_time_s'), ge_metrics.get('wall_time_s')
        ),
    },
    'text_validation': {
        'all_text_healthy': all_text_healthy,
        'eager_ge_similarity_ratio': eager_ge_similarity,
        'eager_npugraph_ex_similarity_ratio': eager_npugraph_similarity,
        'ge_npugraph_ex_similarity_ratio': ge_npugraph_similarity,
        'graph_outputs_consistent': graph_outputs_consistent,
        'all_outputs_consistent': all_outputs_consistent,
    },
    'preferred_graph_backend': preferred_graph_backend,
    'deployment_recommendation': deployment_recommendation,
}
# 先打印紧凑表格，便于在 Notebook 中直观看三种模式。
print(f"{'模式':<16} {'tokens/s':>12} {'decode(s)':>12} {'wall(s)':>12}")
for mode_name, mode_metrics in (
    ('eager', eager_metrics),
    ('ge_graph', ge_metrics),
    ('npugraph_ex', npugraph_metrics),
):
    print(
        f"{mode_name:<16} {mode_metrics['tokens_per_second']:>12.3f} "
        f"{str(mode_metrics.get('decode_time_s')):>12} {mode_metrics['wall_time_s']:>12.3f}"
    )
# 完整结果写入 JSON，方便后续汇总多轮测试或提交实验记录。
comparison_path = SOURCE_DIR / 'run_outputs' / 'graph_backend_comparison.json'
comparison_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print('Eager 文本 =', eager_texts[0] if eager_texts else '')
print('GE Graph 文本 =', ge_texts[0] if ge_texts else '')
print('NPU Graph EX 文本 =', npugraph_texts[0] if npugraph_texts else '')
print('对比文件 =', comparison_path)
if preferred_graph_backend == 'not_determined':
    print('结论：图模式验证或文本检查未通过，暂不选择后端。')
else:
    print('本轮图模式 A/B 中性能更优的后端 =', preferred_graph_backend)
    print('部署建议 =', deployment_recommendation)
